# ATARRA: train the segmentation model on Colab

This notebook trains the 4-class reed segmentation model on real Sentinel-2 imagery of Lake Burullus. It runs in **stages on purpose**:

| Stage | What it does | Cost | How often |
| --- | --- | --- | --- |
| 1. Build | Fetches imagery, tiles it, writes a store to Drive | 30-60 min, ~4 GB of ranged reads | Once |
| 2. Reserve | Exports the tiles a human will label, and excludes their ground from training | seconds | Once, before training |
| 3. Train | Trains from the store | minutes per run | As often as you like |
| 4. Score | Measures the trained model against your labels | seconds | Once you have labelled |

Splitting them is not tidiness. Free-tier Colab sessions cap out and disconnect when idle, so a fetch combined with a training run means a disconnect at 90% throws away both. The store is also the expensive half; training on top of it is cheap and repeatable.

**Stage 2 comes before Stage 3 and that order is load-bearing.** A test set the model has already trained on is not a test set, and the only way to be sure is to reserve the ground first and pass `--exclude-pack` to every training run.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.

### The honest caveat, stated up front

The training labels come from a rule engine (weak supervision), not from field-verified annotation. So the accuracy this notebook prints in Stage 3 measures **agreement with that rule engine** - if the rules are wrong about a pixel, a model that reproduces them faithfully is still scored correct. It is the interim number, and the metrics file says so: `targets.assessable` is `false` and no target verdict is recorded. The proposal's mIoU >= 0.82 claim is substantiated in Stage 4, against labels a human produced, or not at all.


In [ ]:
import sys

print('python ', sys.version.split()[0])

for name in ('numpy',):
    try:
        module = __import__(name)
        print(name.ljust(7), module.__version__)
    except Exception as exc:
        print(name.ljust(7), 'MISSING', exc)

try:
    import torch
    print('torch  ', torch.__version__)
    print('cuda   ', torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print('device ', props.name, str(round(props.total_memory / 1e9, 2)) + ' GB')
    else:
        print()
        print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.')
except Exception as exc:
    print('torch   MISSING', exc)

try:
    import psutil
    print('ram    ', str(round(psutil.virtual_memory().total / 1e9, 1)) + ' GB')
except Exception:
    pass

## Stage 1: build the tile store

Run this section once. It writes to Drive, so a completed store survives the session.

It is **not** incrementally resumable: an interrupted build restarts from the first date, and an incomplete store is detected and discarded rather than built on top of. Colab's free tier disconnects when idle, so start this when you can leave the tab open.

The parameters below are chosen for a free-tier T4:

- **`--gsd 20`** - one date's 8-band composite is 161 MB at 20 m and 644 MB at 10 m, and the whole store is roughly 1.2 GB at 20 m against 5 GB at 10 m. Ten-metre tiles do resolve individual reed stands better, so it is worth revisiting once Drive space and download time are not the scarce things.
- **`--stride 128`** - half the tile size, so tiles overlap 2x. That roughly quadruples the number of training samples for the *same* download, because the imagery is fetched once and cut afterwards. Overlapping tiles are separate samples for the loss, which is the point; what they must never do is appear on both sides of the train/test fence. Stage 3 splits by *ground* for that reason, dropping the tiles that straddle a boundary rather than trusting adjacent tiles to be independent.
- **`--dates 12`** - spread evenly across 24 months rather than the 12 most recent. A model trained only on late summer has seen the one season where reed is easiest to separate from cropland.


In [ ]:
import os
import sys

PROJECT_DIR = '/content/atarra'
DRIVE_ROOT = '/content/drive/MyDrive/atarra'
STORE_DIR = DRIVE_ROOT + '/store_20m'
RUNS_DIR = DRIVE_ROOT + '/runs'
LOCAL_STORE = '/content/store_20m'

# The project's GitHub remote; the setup cell below clones it, so no manual upload.
# Blank it only if you uploaded the project to PROJECT_DIR yourself.
REPO_URL = 'https://github.com/AlimohamedV/atarra.git'

IN_COLAB = 'google.colab' in sys.modules
print('in colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(RUNS_DIR, exist_ok=True)
    print()
    print('store  ->', STORE_DIR)
    print('runs   ->', RUNS_DIR)
else:
    print('not running in Colab; the build and train cells will still work')

if not REPO_URL and not os.path.isdir(PROJECT_DIR) and IN_COLAB:
    print()
    print('Set REPO_URL in this cell to clone the project, or upload it to', PROJECT_DIR)

In [ ]:
import os
import subprocess
import sys

def run(command):
    # Output is captured rather than inherited on purpose: a Colab kernel captures
    # stdout at the Python level, so a child process writing straight to file
    # descriptor 1 prints nothing at all. Silent success then looks exactly like
    # silent failure, which is the worst way for an install step to behave.
    print('$', command, flush=True)
    proc = subprocess.run(command, shell=True, capture_output=True, text=True)
    for stream in (proc.stdout, proc.stderr):
        if stream.strip():
            print(stream.rstrip(), flush=True)
    if proc.returncode != 0:
        raise SystemExit(f'exit code {proc.returncode}')
    return proc

if not os.path.isdir(PROJECT_DIR):
    if not REPO_URL:
        raise SystemExit('Set REPO_URL above, or place the project at ' + PROJECT_DIR)
    run('git clone --depth 1 ' + REPO_URL + ' ' + PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('working directory:', os.getcwd())

# Make the package importable in this kernel immediately.
#
# The editable install writes a .pth file, and a running kernel never re-reads
# those -- so an in-kernel `import atarra` cannot see the install at all. Worse,
# /content is on sys.path in Colab and the clone directory is named atarra, so
# the import does not fail: it silently returns that directory as a namespace
# package, which has no __version__. Putting src/ first resolves both.
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

# requirements-colab.txt deliberately leaves numpy and torch alone. numpy is unpinned
# because the <2 cap in requirements.txt exists for one machine's CUDA torch build,
# and torch is absent because Colab's CUDA build is the reason to be here at all.
run(sys.executable + ' -m pip install -q -r requirements-colab.txt')

# --no-deps is load-bearing: without it, setuptools re-resolves torch and replaces the
# CUDA wheel with whichever build PyPI prefers, and the GPU is gone.
run(sys.executable + ' -m pip install -q -e . --no-deps --no-build-isolation')

run(sys.executable + ' -c "import atarra, rasterio, torch; print(atarra.__version__, rasterio.__version__, torch.__version__)"')

In [ ]:
import json
import os
import shutil
import subprocess
import sys


def store_is_complete(path):
    """Check the shards, not just the manifest.

    A manifest is written at the end of a build, but a disconnected session can leave
    one behind with shards missing or truncated underneath it. Reusing that store would
    fail much later, in the middle of a training run, with an error about a .npy file.
    """
    manifest_path = os.path.join(path, 'manifest.json')
    if not os.path.exists(manifest_path):
        return False, 'no manifest'
    with open(manifest_path) as handle:
        manifest = json.load(handle)
    if not manifest.get('shards'):
        return False, 'the manifest lists no shards'
    for entry in manifest['shards']:
        shard = os.path.join(path, 'shards', entry['date'])
        for name in ('image.npy', 'mask.npy', 'blocks.npy', 'keys.json'):
            if not os.path.exists(os.path.join(shard, name)):
                return False, 'shard ' + entry['date'] + ' is missing ' + name
    return True, str(len(manifest['shards'])) + ' shard(s) present'


complete, why = store_is_complete(STORE_DIR)
if complete:
    print('a complete store already exists at', STORE_DIR)
    print(why + '; nothing to rebuild. Delete that directory to start over.')
else:
    if os.path.exists(STORE_DIR):
        print('the store at', STORE_DIR, 'is incomplete (' + why + '); removing it')
        print('so the rebuild cannot inherit half-written shards.')
        shutil.rmtree(STORE_DIR)
    command = ' '.join([
        sys.executable, '-m', 'atarra.cli', 'dataset', 'build', 'burullus',
        '--out', STORE_DIR,
        '--dates', '12',
        '--months', '24',
        '--max-cloud', '10',
        '--gsd', '20',
        '--stride', '128',
    ])
    print('$', command)
    print()
    print('This performs real reads of the satellite archive. Expect 30-60 minutes.')
    print('It is not resumable: if it is interrupted, run this cell again from the top.')
    print()
    subprocess.run(command, shell=True, check=True)


In [ ]:
from atarra.datasets.store import TileStoreDataset

dataset = TileStoreDataset(STORE_DIR)
info = dataset.describe()

for key in ('area', 'gsd', 'tile_size', 'stride', 'shards', 'tiles', 'scope'):
    print(key.ljust(14), info[key])

print('dates'.ljust(14), len(info['dates']))
print()

names = ['open_water', 'crops_soil', 'mixed_halophytes', 'phragmites']
total = sum(info['class_counts'])
print('class balance over all stored tiles:')
for name, count in zip(names, info['class_counts']):
    share = 100 * count / max(1, total)
    print('  ' + name.ljust(22), str(count).rjust(12), str(round(share, 2)).rjust(7) + '%')

print()
if info['class_counts'][3] == 0:
    print('WARNING: no reed pixels were stored, so the reed class cannot be learned.')
else:
    print('reed is', str(round(100 * info['class_counts'][3] / max(1, total), 2)) + '% of pixels;')
    print('inverse-frequency loss weighting is what stops the model ignoring it.')

print()
print('review share:', info['review_fraction'])
print('(the share of observed ground the rule engine declined to call; it is what the')
print(' annotation pack ranks tiles by, and it counts ground once, not once per tile.)')


## Stage 2: reserve the test ground, then train

The store lives on Drive so it survives, but Drive's FUSE mount is slow for the memory-mapped, seek-heavy reads training does. So the first cell copies it to local disk once; the copy is seconds, and training then reads at full speed.

Then the annotation pack is exported - **before** any training run, because a held-out set the model has already seen is not held out. The exclusion works on ground, not on tile keys: reserving one date's tile reserves that reed bed on every date, and the neighbours within half a tile of it.

Then a **pilot** of a few epochs. A 40-epoch run that turns out to be misconceived costs an hour and tells you nothing; four epochs show whether the loss falls, whether every split has usable labels, and what the split actually did.

Once the pilot looks sane, the two full arms follow:

- **8-band multispectral** - visible, red-edge, NIR and SWIR.
- **RGB control arm** - blue, green, red only.

The proposal claims the multispectral stack beats RGB by 10-15% mIoU. That claim is only substantiated by running both arms on identical data with the same split and seed, which is what these cells do. Both read the same store, so the RGB arm costs no extra download.

Run directories are timestamped, and `atarra train` refuses to write into a directory that already holds a run - two 40-epoch runs whose `metrics.json` overwrote each other would be indistinguishable afterwards. There is no checkpoint-resume path: an interrupted run restarts. The store is what makes that cheap.


In [ ]:
import os
import shutil

if os.path.exists(LOCAL_STORE + '/manifest.json'):
    print('local copy already present at', LOCAL_STORE)
else:
    print('copying', STORE_DIR, '->', LOCAL_STORE)
    shutil.copytree(STORE_DIR, LOCAL_STORE)

size = 0
for root, dirs, files in os.walk(LOCAL_STORE):
    for name in files:
        size += os.path.getsize(os.path.join(root, name))

print('local store size:', str(round(size / 1e6, 1)) + ' MB')


In [ ]:
import json
import os
import subprocess
import sys

PACK_DIR = DRIVE_ROOT + '/annotation_pack'
PACK_MANIFEST = os.path.join(PACK_DIR, 'pack.json')

if os.path.exists(PACK_MANIFEST):
    with open(PACK_MANIFEST) as handle:
        existing = json.load(handle)
    print('annotation pack already reserved:', len(existing['reserved_keys']), 'tile(s)')
    print('these keys, and the same ground on every other date, are excluded from training.')
else:
    command = ' '.join([
        sys.executable, '-m', 'atarra.cli', 'annotation', 'export', LOCAL_STORE,
        '--out', PACK_DIR,
        '--limit', '20',
        '--strategy', 'reed',
    ])
    print('$', command)
    subprocess.run(command, shell=True, check=True)

with open(PACK_MANIFEST) as handle:
    pack = json.load(handle)

print()
print('reserved tiles   :', len(pack['reserved_keys']))
print('reed px available:', pack['reed_px_total'])
if not pack['reed_px_sufficient']:
    print()
    print('WARNING: too few reed pixels to measure a reed IoU. Raise --limit, or widen')
    print('the store; the per-class number would be noise otherwise.')

print()
print('Strategy "reed" deliberately over-samples the class the proposal quotes a number')
print('for, so the test set measures reed rather than the area average. An "uncertainty"')
print('pack would instead be a hard-case set - defensible, but it is a different claim.')


In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone

STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')


def train(bands, label, epochs, *, stamp=None):
    """One run. The name carries a timestamp so two runs cannot share a directory."""
    name = 'unet_' + label + '_' + (stamp or STAMP)
    command = ' '.join([
        sys.executable, '-m', 'atarra.cli', 'train', LOCAL_STORE,
        '--bands', bands,
        '--epochs', str(epochs),
        '--batch-size', '16',
        '--workers', '2',
        '--out', RUNS_DIR,
        '--name', name,
        '--exclude-pack', PACK_DIR,
    ])
    print('$', command)
    print()
    subprocess.run(command, shell=True, check=True)
    print()
    return name


pilot = train('8', 'pilot', 4)

with open(os.path.join(RUNS_DIR, pilot, 'metrics.json')) as handle:
    metrics = json.load(handle)

splits = metrics['splits']
print('train / val / test tiles :', splits['train_tiles'], '/', splits['val_tiles'],
      '/', splits['test_tiles'])
print('boundary tiles omitted   :', splits['boundary_tiles_omitted'],
      '(too close to a split edge to be on either side)')
print('split gap used           :', splits['buffer_pixels'], 'px',
      '(narrower than requested means this store could not afford the default)')
print('geographic split verified:', splits['verified_disjoint'])
stored = metrics['dataset']['tiles']
held = metrics['holdout']['tiles_excluded_geographically']
print('held-out ground          :', metrics['holdout']['tiles_reserved'], 'reserved tile(s)',
      'expanding to', held, 'of', stored,
      '(' + str(round(100 * held / max(1, stored))) + '% of the store)')
if held > 0.25 * stored:
    print('  NOTE: that is a large share of this store. Reserving ground removes')
    print('  the same reed bed on every date, and its neighbours with it, so a')
    print('  20-tile pack can take a quarter of the training data on a small store.')
print('weights evaluated        : epoch', metrics['training']['evaluated_epoch'],
      'of', metrics['training']['epochs_run'], '(the best one, not the last)')
print()
print('class support per split (tile pixels, so overlapping tiles count twice):')
for name in ('train', 'val', 'test'):
    print('  ' + name.ljust(6), splits['class_support'][name])
print()
print('targets assessable:', metrics['targets']['assessable'])
print(metrics['targets']['reason'])


In [ ]:
# Both arms cut from the same store, with the same split and seed, so the only
# difference between them is the band set - which is what the 10-15% claim is about.
eight_name = train('8', '8band', 40)
rgb_name = train('rgb', 'rgb', 40)


In [ ]:
import json
import os


def latest(label):
    """The newest run of one arm, found on disk so this cell works in a later session."""
    prefix = 'unet_' + label + '_'
    candidates = sorted(name for name in os.listdir(RUNS_DIR) if name.startswith(prefix))
    if not candidates:
        raise SystemExit('no completed run of ' + label + ' in ' + RUNS_DIR)
    return candidates[-1]


def metrics(label):
    with open(os.path.join(RUNS_DIR, latest(label), 'metrics.json')) as handle:
        return json.load(handle)


eight = metrics('8band')
rgb = metrics('rgb')

print('arm'.ljust(10), 'mIoU'.rjust(8), 'reed IoU'.rjust(10), 'reed F1'.rjust(9), 'pixel acc'.rjust(11))
print('-' * 50)
for label, data in (('8-band', eight), ('rgb', rgb)):
    report = data['test_report']
    print(label.ljust(10),
          str(report['mean_iou']).rjust(8),
          str(report['phragmites_iou']).rjust(10),
          str(report['phragmites_f1']).rjust(9),
          str(report['pixel_accuracy']).rjust(11))

print()
print('per-class, 8-band:')
for entry in eight['test_report']['per_class']:
    print('  ' + entry['class_name'].ljust(24),
          'IoU', str(entry['iou']).rjust(8),
          'F1', str(entry['f1']).rjust(8),
          'support', str(entry['support_px']).rjust(10))

gain = eight['test_report']['mean_iou'] - rgb['test_report']['mean_iou']
base = rgb['test_report']['mean_iou']

print()
print('absolute gain:', str(round(gain, 4)), 'mIoU in favour of the multispectral arm')
print()

# A relative gain is only interpretable against a baseline that is actually
# measuring something. An earlier version of this cell guarded with `base > 0`, and
# since a barely-trained control arm scores ~0.0002 that guard passes and the cell
# reports 62850% -- a number that looks like a spectacular result and means nothing.
# The proposal's 10-15% claim needs both arms trained to convergence first.
MIN_INTERPRETABLE_BASELINE = 0.05
if base >= MIN_INTERPRETABLE_BASELINE:
    print('relative gain:', str(round(100 * gain / base, 1)) + '%', '(proposal target: 10-15%)')
else:
    print('relative gain NOT reported.')
    print('The RGB control arm scored', str(round(base, 4)) + ', which is too close to')
    print('zero for a ratio to be meaningful. Train both arms to convergence first.')

print()
print('CAVEAT:')
print(eight['labels']['caveat'])


## Stage 4: the number that can be defended

Everything above measures the model against the rule engine it was trained on. The annotation pack reserved in Stage 2 is the alternative: a block of pixels a human read, on ground the model never saw.

The tiles were chosen by the rule engine's own uncertainty, ranked so that the tiles where a human decision buys the most accuracy per minute come first. Each chip ships with the rule engine's current guess, so you **correct** a guess rather than starting from a blank raster - and with a review mask showing exactly which pixels it declined to call.

Annotating the entire pack is usually not realistic: on real Burullus imagery a majority of pixels land in the review queue, because spectrally mixed ground genuinely is ambiguous. Annotate as many tiles as you can, then score. The scorer refuses to publish a verdict it cannot support - it checks that the annotation covers more than one class, that enough reed is annotated to measure, and that the checkpoint records no annotated tile in its training or validation splits - and prints `NOT ASSESSED` with the reason otherwise.


In [ ]:
import json
import os
import sys

with open(os.path.join(PACK_DIR, 'pack.json')) as handle:
    pack = json.load(handle)

run = latest('8band')

print('reserved tiles :', len(pack['reserved_keys']))
print('chips to label :', PACK_DIR + '/chips')
print('run to score   :', os.path.join(RUNS_DIR, run, 'best.pt'))
print()
print('1. open the chips in QGIS and label each one. Save as labels/<stem>.tif on the')
print('   chip grid, or as labels/<stem>.geojson with an integer class_code field; the')
print('   codes are in the README that was written beside them. The labels/ directory')
print('   is empty until you do - scoring without it fails on purpose rather than')
print('   quietly falling back to the rule engine.')
print()
print('2. score the trained model against your labels:')
print()
print('   ' + sys.executable + ' -m atarra.cli annotation score ' + PACK_DIR +
      ' --checkpoint ' + os.path.join(RUNS_DIR, run, 'best.pt'))
print()
print('A run written before this notebook reserved anything carries no provenance, and')
print('the scorer will say the independence of the score is unverifiable rather than')
print('assume it. Retraining with --exclude-pack is what makes that check pass.')


In [ ]:
import os

print('artifacts under', RUNS_DIR)
print()
for root, dirs, files in os.walk(RUNS_DIR):
    for name in sorted(files):
        path = os.path.join(root, name)
        print(str(round(os.path.getsize(path) / 1e6, 2)).rjust(8) + ' MB  ' + path)

print()
print('Each run directory holds best.pt (weights, the normalisation buffers so inference')
print('cannot use different statistics from training, the ordered band names, and the')
print('record of which tiles went to which split), metrics.json, and history.json. The')
print('provenance is what lets the scorer check that annotated ground was never trained')
print('on, so it is worth keeping even after the run is superseded.')
